# MissionCtrl — One-Click Training Notebook

**OpenEnv Hackathon Round 2**

This notebook trains an OverseerAgent to detect hallucinations in a multi-agent fleet using GRPO + Unsloth.

**Runtime**: **Kaggle** 2×T4 (see section below).  
**Base model (`train.py`)**: default `Qwen/Qwen2.5-0.5B-Instruct` (fast canary). For larger hubs: `MISSIONCTRL_MODEL_NAME=unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit` (or `unsloth/Llama-3.2-3B-Instruct`). Gated models: `HF_TOKEN` + license on HuggingFace.  
**Expected training time**: 0.5B is much faster than 3B/8B; full curriculum on Kaggle 2×T4 can still take hours if phases repeat; `train.py` shortens steps when two GPUs are visible.  
**Expected final reward**: 0.68+ (reward ceiling is **0.85**, not 1.0)  

## Bug Fixes Applied
- ✅ **#1** GRPO reward fn now runs full episode rollout (not single step)
- ✅ **#2** NUM_GENERATIONS ≤ BATCH_SIZE (was 8 > 4, caused crash)
- ✅ **#3** evaluate() uses final reward, not accumulated sum of state scores
- ✅ **#4** run_baseline() uses final reward (not per-task overwrite)
- ✅ **#5** Reward ceiling corrected to 0.85 throughout
- ✅ **#6** SYNTHESIZE_REPORT() now promotes PENDING tasks too, terminates episode
- ✅ **#7** 'special' difficulty tier now supported
- ✅ **#8** All 10 hallucination types documented and in system prompt
- ✅ **#10** SUBTLETY levels now affect corruption templates
- ✅ **#11** tokenizer.padding_side='left' set for correct GRPO generation
- ✅ **#12** plot_reward_curve() guards against empty history
- ✅ **#13** Easy phase uses 3 tasks (consistent with docs)
- ✅ **#15** _corrupt() no longer relies on 'def ' being present
- ✅ **#16** All bare except: changed to except Exception:
- ✅ **#17** FAILED status now assigned via ESCALATE action
- ✅ **#20** Reward function exceptions now logged (not silently swallowed)
- ✅ **#21** 2+ GPU: shorter Kaggle curriculum; model stays on one CUDA device by default for GRPO generation stability
- ✅ **#22** max_prompt_length set in GRPOConfig to prevent seed tag truncation
- ✅ **#23** evaluate() uses do_sample=False for truly greedy/reproducible eval
- ✅ **#25** Passive FP penalty only fires when all task outputs are visible
- ✅ **#26** EVIDENCE_KEYWORDS uses specific phrases, not generic single words
- ✅ **#27** _build_observation() de-duplicates messages per task_id
- ✅ **#28** Hub push and local save both use LoRA format consistently
- ✅ **#29** DEPENDENCY_MAP variable naming clarified
- ✅ **#30** SubAgentSimulator seed offset +7 documented
- ✅ **#32** HF_REPO validation strengthened (case-insensitive)
- ✅ **#35** Model always restored to training mode after eval
- ✅ **#38** System prompt lists all 10 hallucination types
- ✅ **#39** EPISODE_MAX_STEPS constant used consistently across training and eval
- ✅ **#40** Fake credential strings use clearly-invalid tokens

## Kaggle (2×T4)

- **Ways to get the code (the next cell checks in this order):**  
  1) **Same directory as the notebook** (Kaggle kernel push, or files uploaded to working space).  
  2) **Kaggle "Add data" (Input):** if you attach a dataset, files live at `/kaggle/input/<your-dataset-slug>/` (read-only). The next cell **detects** that path and `cd` there when all required `.py` files are present.  
  3) If still missing, it **clones** the [default training repo](https://github.com/Leo-Expose/MissionCtrl) to `/kaggle/working/MissionCtrl/`. Override the clone URL with **`MISSIONCTRL_REPO_URL`** in **Add-ons → Environment** if you use another fork. Private git: use a [Kaggle dataset](https://www.kaggle.com/datasets) you build from the repo, or a git URL (see README: **Kaggle training and Kaggle CLI**). Do not paste secrets in the notebook.
- **Notebook settings**: accelerator **2 × T4**; **Internet** ON (required for `pip` and for `git clone` in the default flow). **Add-ons → Secrets**: `HF_TOKEN` (gated Unsloth bases: accept the license on HuggingFace first).
- **Checkpoints** on Kaggle: `train.py` uses `/kaggle/working/missionctrl_checkpoints` when appropriate.
- **Run training** after the clone/verify cell: `!python train.py` or `!python train.py --smoke-train`. **2 GPUs**: shorter curriculum in `train.py`; the model loads on one CUDA device by default to avoid GRPO generation device-mismatch crashes.
- **MCP:** no Kaggle launcher in this repo; see the README.


In [ ]:
# ── Install dependencies (Kaggle) ────────────────────────────────────────────
# unsloth[colab-new] works on many Kaggle images; if install fails, see Unsloth docs.
# On the Kaggle image, pip may show resolver messages (e.g. fsspec/s3fs); if installs
# complete and the next cell runs, those are usually safe to ignore.
!pip install "unsloth[colab-new]" trl openenv transformers datasets accelerate matplotlib --quiet
!pip install --upgrade bitsandbytes --quiet

# Suppress tokenizer parallelism warning
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Dependencies installed')

In [ ]:
# ── Clone (if needed) + verify MissionCtrl project files ────────────────────
# 1) Same folder as the notebook; 2) /kaggle/input/<dataset>/ if you "Add data"; 3) then git clone.
# Set MISSIONCTRL_REPO_URL in Add-ons → Environment to override the default public repo.
import os
import subprocess

REQUIRED = [
    "train.py",
    "environment.py",
    "reward_model.py",
    "grpo_rewards.py",
    "grpo_completion.py",
]
REPO_DIR = "MissionCtrl"
DEFAULT_REPO = "https://github.com/Leo-Expose/MissionCtrl.git"
REPO_URL = os.environ.get("MISSIONCTRL_REPO_URL", DEFAULT_REPO).strip() or DEFAULT_REPO


def all_here() -> bool:
    return all(os.path.isfile(f) for f in REQUIRED)


def all_here_in(d: str) -> bool:
    return all(os.path.isfile(os.path.join(d, f)) for f in REQUIRED)


def missing_in_dir(d: str) -> list:
    return [f for f in REQUIRED if not os.path.isfile(os.path.join(d, f))]


# Kaggle: files are often under /kaggle/input/<dataset_name>/ (read-only) — chdir so imports work
if not all_here() and os.path.isdir("/kaggle/input"):
    for name in sorted(os.listdir("/kaggle/input")):
        root = os.path.join("/kaggle/input", name)
        if not os.path.isdir(root) or name.startswith("."):
            continue
        if all_here_in(root):
            os.chdir(root)
            print(f"✅ Using Kaggle input dataset: {root}")
            break
        nest = os.path.join(root, REPO_DIR)
        if os.path.isdir(nest) and all_here_in(nest):
            os.chdir(nest)
            print(f"✅ Using nested folder in input: {nest}")
            break

if all_here():
    print("✅ MissionCtrl files already in this directory (kernel bundle or /kaggle/input). Skipping git clone.")
else:
    is_kaggle = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle/working")
    if is_kaggle and os.path.isdir("/kaggle/working"):
        os.chdir("/kaggle/working")
    in_repo = all(os.path.isfile(os.path.join(REPO_DIR, f)) for f in REQUIRED)
    if in_repo:
        os.chdir(REPO_DIR)
    elif not all_here():
        if os.path.isdir(REPO_DIR):
            _miss = ", ".join(missing_in_dir(REPO_DIR)) or "(unknown)"
            raise FileNotFoundError(
                f"Directory {REPO_DIR!r} exists but is incomplete (missing: {_miss}). "
                f"Delete it in the file browser and re-run, or set MISSIONCTRL_REPO_URL. "
                f"See README: Kaggle training and Kaggle CLI (Troubleshooting)."
            )
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True,
        )
        os.chdir(REPO_DIR)
    if not all_here():
        _miss = ", ".join(missing_in_dir(".")) or ", ".join(REQUIRED)
        raise FileNotFoundError(
            f"Could not find train.py and env modules (missing: {_miss}). Set MISSIONCTRL_REPO_URL to a public "
            f"git URL (default {DEFAULT_REPO!r}) or add a Kaggle dataset with the repo. "
            "See README: Kaggle training and Kaggle CLI (Troubleshooting)."
        )

for f in REQUIRED:
    print("  ✅", f)
print("  cwd:", os.getcwd())

# Jupyter keeps the notebook directory on sys.path, not cwd — add repo root for local imports.
import sys
_repo_root = os.path.abspath(".")
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)


In [ ]:
# ── Cell 3: Verify GPU ────────────────────────────────────────────────────────
import torch

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'GPU count: {torch.cuda.device_count()}')
assert torch.cuda.is_available(), "No GPU — switch Kaggle accelerator to GPU (2×T4 recommended)"

# 2×GPU / T4: train.py auto-tunes the Kaggle curriculum while keeping GRPO generation on one device.
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
n_gpu = torch.cuda.device_count()
if n_gpu >= 2:
    print(f"✅ {n_gpu} GPUs — Kaggle 2×T4-style path: shorter curriculum; single-device GRPO generation")
elif vram_gb < 30:
    print(f"⚠️  ~{vram_gb:.0f} GB VRAM (e.g. single T4) — use MISSIONCTRL_T4_CURRICULUM=1 for shorter phases if needed")
else:
    print(f"✅ ~{vram_gb:.0f} GB — single-GPU fallback curriculum")

print('✅ GPU verified')

In [ ]:
# ── Cell 4: Smoke-test the environment ───────────────────────────────────────
from environment import MissionCtrlEnv, OverseerAction, parse_action
from reward_model import compute_reward, reward_breakdown

print('--- Smoke test: medium difficulty, 3 tasks ---')
env = MissionCtrlEnv(difficulty='medium', num_tasks=3, seed=42)
obs, info = env.reset()

print(f'  Tasks loaded: {len(obs["task_board"])}')
print(f'  Agent messages: {len(obs["recent_messages"])}')

# Test FLAG action
first_task = obs['task_board'][0]['task_id']
action = OverseerAction('FLAG', task_id=first_task, evidence='fabricated citation detected in prior output')
obs2, reward, terminated, truncated, info = env.step(action)
print(f'  Step reward: {reward:.3f}')
print(f'  Info: {info}')

# Test reward breakdown
breakdown = reward_breakdown(env)
print(f'  Reward breakdown: {breakdown["signals"]}')
print(f'  Theoretical ceiling: {breakdown["theoretical_ceiling"]}')

print()
print('--- Smoke test: special difficulty (new tier) ---')
env_special = MissionCtrlEnv(difficulty='special', num_tasks=5, seed=99)
obs_s, _ = env_special.reset()
print(f'  Special tier tasks: {len(obs_s["task_board"])}')

print()
print('--- Smoke test: action parser ---')
test_cases = [
    ('FLAG(T001, "fabricated citation in reference list")', 'FLAG'),
    ('APPROVE(T002)', 'APPROVE'),
    ('SYNTHESIZE_REPORT()', 'SYNTHESIZE'),
    ('garbage output no action', 'NOOP'),
]
for text, expected in test_cases:
    parsed = parse_action(text)
    status = '✅' if parsed.action_type == expected else f'❌ expected {expected}'
    print(f'  {status} "{text[:40]}..." → {parsed.action_type}')

print()
print('✅ Environment smoke test passed')

In [ ]:
# ── Cell 5: Run pre-training baseline ────────────────────────────────────────
# FIX #3/#4: baseline now uses final reward (not accumulated sum or per-task overwrite)
from train import run_baseline

baseline_reward = run_baseline()
print(f'\n🎯 Baseline established: {baseline_reward:.3f}')
print(f'Reward ceiling: 0.85 | Training target: 0.68+')
print(f'Baseline is {baseline_reward/0.85*100:.0f}% of theoretical ceiling')

In [ ]:
# ── Cell 6: Set HuggingFace credentials ──────────────────────────────────────
from huggingface_hub import login
import os

# Option A: Use Kaggle Secrets (recommended)
# 1. Go to Add-ons → Secrets
# 2. Add a new secret named "HF_TOKEN" with your HuggingFace token
# 3. Run this cell:
if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'])
    print('✅ Logged in via HF_TOKEN secret')
else:
    # Option B: Interactive prompt
    login()
    print('✅ Logged in interactively')

# Set your repo name before training
# FIX #32: validation is now case-insensitive and catches more placeholder patterns
import train
train.HF_REPO = 'Proliferation/missionctrl_env'  # ← CHANGE THIS

# Verify it's been updated
if 'your-hf' in train.HF_REPO.lower():
    raise ValueError('❌ Please update train.HF_REPO with your actual HuggingFace username!')

print(f'✅ Will push trained adapter to: {train.HF_REPO}')

# ── IMPORTANT: Verify evaluation uses your model ──────────────────────────────
print()
print('⚠️  IMPORTANT: Before training, confirm the hackathon evaluation invokes')
print('   your HF-hosted model (not a fixed Groq/API endpoint).')
print('   If evaluators run client.py with their own API_BASE_URL, fine-tuning')
print('   will not affect your score. Check the OpenEnv evaluation spec first.')

In [ ]:
# ── Cell 7: TRAIN ─────────────────────────────────────────────────────────────
# Full 3-phase curriculum with reward-gated advancement.
#
# Key fixes active in this run:
#   - NUM_GENERATIONS=4 ≤ BATCH_SIZE=4 (no crash)
#   - Full episode rollout in reward fn (meaningful GRPO signal)
#   - max_prompt_length set (seed tag won't be truncated)
#   - padding_side='left' (correct batched generation)
#   - Greedy eval (do_sample=False) for reproducible metrics
#
# Checkpoints: saved to train.OUTPUT_DIR every 50 steps (Kaggle: /kaggle/working/missionctrl_checkpoints).
# If training is interrupted, resume from the last checkpoint.

from train import train

history = train()
print('\n🏆 Training complete!')

In [ ]:
# ── Cell 8: Display reward curve ─────────────────────────────────────────────
# FIX #5: curve now shows 0.85 ceiling line (not 1.0)
# FIX #12: plot_reward_curve() no longer crashes on empty history
from IPython.display import Image
import os

import train
curve_path = os.path.join(train.OUTPUT_DIR, 'reward_curve.png')

if os.path.exists(curve_path):
    Image(curve_path)
else:
    print('Reward curve not found at expected path.')
    print(f'Check: {train.OUTPUT_DIR}')
    # Try local fallback
    local_path = './missionctrl_checkpoints/reward_curve.png'
    if os.path.exists(local_path):
        print(f'Found at local path: {local_path}')
        Image(local_path)

In [ ]:
# ── Cell 9: Before/After demo comparison ─────────────────────────────────────
# FIX #28: load LoRA adapter (not merged model) — consistent with what was pushed to Hub
from unsloth import FastLanguageModel
from environment import MissionCtrlEnv, parse_action
from train import build_user_prompt, SYSTEM_PROMPT, load_model
import torch
import os

# Load base model + LoRA adapter
import train
adapter_path = os.path.join(train.OUTPUT_DIR, 'final_lora')
if not os.path.exists(adapter_path):
    adapter_path = os.path.join('missionctrl_checkpoints', 'final_lora')

model, tokenizer = load_model()   # loads base model with LoRA config
model.load_adapter(adapter_path)  # loads trained adapter weights
FastLanguageModel.for_inference(model)

# Use a known hallucinated episode (seed 0, hard difficulty)
env = MissionCtrlEnv(difficulty='hard', num_tasks=4, seed=0)
obs, _ = env.reset()

print('=== ENVIRONMENT STATE ===')
print(env.render())

prompt = tokenizer.apply_chat_template(
    [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': build_user_prompt(obs)},
    ],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=3584).to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,   # FIX #23: greedy for demo reproducibility
    )

completion = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('\n=== TRAINED MODEL OUTPUT ===')
print(completion)

parsed = parse_action(completion)
print(f'\nParsed action: {parsed.action_type}(task_id={parsed.task_id}, evidence={parsed.evidence!r})')

In [ ]:
# ── Cell 10: Final evaluation run ────────────────────────────────────────────
# FIX #3: rewards are now final-step state scores, not accumulated sums
# FIX #23: uses do_sample=False for reproducible metrics
from train import evaluate

final_reward, metrics = evaluate(
    model, tokenizer,
    difficulty='hard',
    num_tasks=4,
    n_episodes=20,
)

print('\n=== FINAL EVALUATION SUMMARY ===')
print(f'  Overall reward:        {metrics["mean_reward"]:.3f} ± {metrics["std_reward"]:.3f}')
print(f'  % of ceiling (0.85):  {metrics["mean_reward"]/0.85*100:.1f}%')
print(f'  Detection rate:        {metrics["mean_detect_rate"]:.1%}')
print(f'  False positive rate:   {metrics["mean_fp_rate"]:.1%}')

if metrics['mean_reward'] >= 0.68:
    print(f'\n✅ Target of 0.68+ achieved!')
else:
    print(f'\n⚠️  Below 0.68 target. Consider: more training steps, prompt tuning, or policy refinement.')

In [ ]:
# ── Cell 11: Multi-tier evaluation (all difficulties) ────────────────────────
# Evaluate across all difficulty tiers to check for tier-specific regressions
# (Council noted: fast-closure policy improved Hard but hurt Special in baseline)
from train import evaluate

tiers = [
    ('easy',    2, 10),
    ('medium',  3, 10),
    ('hard',    4, 10),
    ('special', 5, 10),
]

print('=== MULTI-TIER EVALUATION ===')
print(f'{"Tier":<10} {"Reward":<12} {"Detect%":<12} {"FP%":<10} {"% Ceiling"}')
print('-' * 55)

for difficulty, num_tasks, n_eps in tiers:
    reward, m = evaluate(
        model, tokenizer,
        difficulty=difficulty,
        num_tasks=num_tasks,
        n_episodes=n_eps,
    )
    pct = reward / 0.85 * 100
    print(f'{difficulty:<10} {reward:.3f} ± {m["std_reward"]:.3f}  '
          f'{m["mean_detect_rate"]:.1%}       {m["mean_fp_rate"]:.1%}      {pct:.0f}%')

In [ ]:
# ── Cell 12: Reward breakdown for a single episode ───────────────────────────
# Useful for debugging which signals are limiting performance
from environment import MissionCtrlEnv, TaskStatus
from reward_model import reward_breakdown
from train import build_user_prompt, SYSTEM_PROMPT, EPISODE_MAX_STEPS
import torch

debug_env = MissionCtrlEnv(difficulty='hard', num_tasks=4, seed=42)
obs, _ = debug_env.reset()

done = False
steps = 0
while not done and steps < EPISODE_MAX_STEPS:
    prompt_text = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user',   'content': build_user_prompt(obs)}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True,
                       max_length=3584).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    completion = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    from environment import parse_action
    action = parse_action(completion)
    obs, _, terminated, truncated, _ = debug_env.step(action)
    done = terminated or truncated
    steps += 1

bd = reward_breakdown(debug_env)
print('=== REWARD BREAKDOWN (seed=42, hard, 4 tasks) ===')
print(f'Total reward: {bd["total_reward"]:.4f} ({bd["total_reward"]/0.85*100:.1f}% of ceiling)')
print()
for signal, vals in bd['signals'].items():
    bar = '█' * int(vals['raw'] * 20)
    print(f'  {signal:<28} raw={vals["raw"]:.3f} [{bar:<20}] weighted={vals["weighted"]:+.4f}')
print()
print(f'Episode info: {bd["info"]}')